In [9]:
import time
import torch
import torchmetrics
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from torch.utils.data import DataLoader, TensorDataset


def train(
    model, optimizer, criterion, metric,
    train_loader, valid_loader, n_epochs,
    warmup_scheduler=None, scheduler=None, patience=None,
    checkpoint_path='best_model.pt', clip_grad_norm=None, device='cpu'
):
    history = {
        'train_losses': [], 'train_metrics': [],
        'valid_metrics': [], 'learning_rates': [],
    }
    best_epoch = 0
    best_valid_metric = float('-inf') if metric.higher_is_better else float('inf')
    patience_counter = 0

    for epoch in range(n_epochs):
        if warmup_scheduler is not None:
            warmup_scheduler.step()

        losses = []
        metric.reset()
        model.train()
        t0 = time.time()

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            losses.append(loss.item())
            loss.backward()
            if clip_grad_norm is not None:
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad_norm)
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred.softmax(dim=1)[:, 1], y_batch)

        train_loss = np.mean(losses)
        train_metric = metric.compute().item()

        model.eval()
        metric.reset()
        with torch.no_grad():
            for X_batch, y_batch in valid_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                y_pred = model(X_batch)
                metric.update(y_pred.softmax(dim=1)[:, 1], y_batch)

        valid_metric = metric.compute().item()

        is_best = (valid_metric > best_valid_metric) if metric.higher_is_better else (valid_metric < best_valid_metric)
        if is_best:
            torch.save(model.state_dict(), checkpoint_path)
            best_valid_metric = valid_metric
            best_epoch = epoch + 1

        if scheduler is not None:
            if epoch >= (warmup_scheduler.total_iters if warmup_scheduler is not None else 0):
                if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                    scheduler.step(valid_metric)
                else:
                    scheduler.step()
        learning_rate = optimizer.param_groups[0]["lr"]

        history['train_losses'].append(train_loss)
        history['train_metrics'].append(train_metric)
        history['valid_metrics'].append(valid_metric)
        history['learning_rates'].append(learning_rate)

        print(
            f'Epoch: {epoch+1}/{n_epochs}, '
            f'Train Loss: {round(train_loss,3)}, '
            f'Train Acc: {round(train_metric,3)}, '
            f'Valid Acc: {round(valid_metric,3)}, '
            f'LR: {learning_rate:.5f}, '
            f'Time: {round(time.time()-t0,2)}s'
            + (' ← best' if is_best else '')
        )

        if patience is not None:
            if is_best:
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch+1}. Best epoch: {best_epoch}")
                    break

    print(f"\nRestoring best model from epoch {best_epoch} (valid acc: {best_valid_metric:.4f})")
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    return history


def plot_history(history, save_path='training_history.png'):
    n_epochs = len(history['train_metrics'])
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle('Heart Disease Classifier – Training History', fontsize=14, fontweight='bold')

    ax = axes[0]
    ax.plot(range(1, n_epochs+1), history['train_losses'], '--r.', label='Train Loss')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title('Training Loss')
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(range(1, n_epochs+1), history['train_metrics'], '--r.', label='Train')
    ax.plot(range(1, n_epochs+1), history['valid_metrics'], '--b.', label='Valid')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy'); ax.set_title('Train vs Valid Accuracy')
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[2]
    ax.plot(range(1, n_epochs+1), history['learning_rates'], '--g.', label='LR')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Learning Rate'); ax.set_title('Learning Rate Schedule')
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')


df = pd.read_csv('cleanned.csv')
print(f"\nDataset shape: {df.shape}")
print(f"Target distribution:\n{df['num'].value_counts().sort_index()}")


cat_cols = ['sex', 'cp', 'restecg']
bool_cols = ['fbs', 'exang']

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

for col in bool_cols:
    df[col] = df[col].map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)


df['label'] = (df['num'] > 0).astype(int)

feature_cols = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalch', 'exang', 'oldpeak']
X = df[feature_cols].values.astype(np.float32)
y = df['label'].values.astype(np.int64)


X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test     = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f"\nSplit sizes — Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

def make_loader(X, y, batch_size=64, shuffle=False):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                       torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader   = make_loader(X_val,   y_val)
test_loader  = make_loader(X_test,  y_test)

class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.act = nn.ReLU()

    def forward(self, x):
        return self.act(x + self.block(x))


class HeartDiseaseNet(nn.Module):
    def __init__(self, in_features=10, hidden_dim=128, n_blocks=3, n_classes=2, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.res_blocks = nn.Sequential(*[ResidualBlock(hidden_dim, dropout) for _ in range(n_blocks)])
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, n_classes),
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.res_blocks(x)
        return self.classifier(x)


device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(42)
model = HeartDiseaseNet(in_features=10, hidden_dim=128, n_blocks=3, n_classes=2, dropout=0.3).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")

class_counts = np.bincount(y_train)
weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=80, eta_min=1e-5)

class AccuracyMetric(torchmetrics.Accuracy):
    higher_is_better = True

metric = AccuracyMetric(task='binary').to(device)

print("Training")

history = train(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    metric=metric,
    train_loader=train_loader,
    valid_loader=val_loader,
    n_epochs=30,
    scheduler=scheduler,
    patience=6,
    checkpoint_path='best_heart_model.pt',
    clip_grad_norm=1.0,
    device=device,
)

print("Test Evalitaion")

from torchmetrics.classification import BinaryAccuracy, BinaryF1Score, BinaryAUROC, BinaryConfusionMatrix

model.eval()
all_preds, all_probs, all_labels = [], [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        logits = model(X_b.to(device))
        probs  = torch.softmax(logits, dim=1)[:, 1]
        preds  = logits.argmax(dim=1)
        all_preds.append(preds.cpu())
        all_probs.append(probs.cpu())
        all_labels.append(y_b)

all_preds  = torch.cat(all_preds)
all_probs  = torch.cat(all_probs)
all_labels = torch.cat(all_labels)

acc  = BinaryAccuracy()(all_preds, all_labels).item()
f1   = BinaryF1Score()(all_preds, all_labels).item()
auroc = BinaryAUROC()(all_probs, all_labels).item()
cm   = BinaryConfusionMatrix()(all_preds, all_labels).numpy()

print(f"Accuracy : {acc:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUROC    : {auroc:.4f}")
print(f"\nConfusion Matrix (rows=actual, cols=pred):")
print(f"           Pred 0   Pred 1")
print(f"Actual 0:  {cm[0,0]:5d}    {cm[0,1]:5d}")
print(f"Actual 1:  {cm[1,0]:5d}    {cm[1,1]:5d}")


plot_history(history, save_path='training_history.png')


fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'Test Set Results  |  Acc={acc:.3f}  F1={f1:.3f}  AUROC={auroc:.3f}', fontsize=13, fontweight='bold')

im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['No Disease', 'Disease'])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['No Disease', 'Disease'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i, j], ha='center', va='center',
                     color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=16, fontweight='bold')

from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(all_labels.numpy(), all_probs.numpy())
axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC={auroc:.3f})')
axes[1].plot([0,1],[0,1],'k--', alpha=0.4, label='Random')
axes[1].fill_between(fpr, tpr, alpha=0.1)
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('test_results.png', dpi=150, bbox_inches='tight')


Dataset shape: (918, 11)
Target distribution:
num
0    410
1    265
2    108
3    107
4     28
Name: count, dtype: int64

Split sizes — Train: 642, Val: 138, Test: 138
Model parameters: 110,658
Training
Epoch: 1/30, Train Loss: 0.584, Train Acc: 0.664, Valid Acc: 0.754, LR: 0.00100, Time: 0.26s ← best
Epoch: 2/30, Train Loss: 0.527, Train Acc: 0.782, Valid Acc: 0.746, LR: 0.00100, Time: 0.19s
Epoch: 3/30, Train Loss: 0.423, Train Acc: 0.779, Valid Acc: 0.739, LR: 0.00100, Time: 0.19s
Epoch: 4/30, Train Loss: 0.426, Train Acc: 0.808, Valid Acc: 0.79, LR: 0.00099, Time: 0.18s ← best
Epoch: 5/30, Train Loss: 0.513, Train Acc: 0.807, Valid Acc: 0.79, LR: 0.00099, Time: 0.25s
Epoch: 6/30, Train Loss: 0.397, Train Acc: 0.812, Valid Acc: 0.761, LR: 0.00099, Time: 0.22s
Epoch: 7/30, Train Loss: 0.445, Train Acc: 0.808, Valid Acc: 0.804, LR: 0.00098, Time: 0.21s ← best
Epoch: 8/30, Train Loss: 0.535, Train Acc: 0.799, Valid Acc: 0.804, LR: 0.00098, Time: 0.21s
Epoch: 9/30, Train Loss: 0.544, T